In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader
from typing import List
from langchain_classic.schema import Document

In [ ]:
file_path = Path(r"C:\Users\amanm\Desktop\learning\rag-project-2\data\2506.08276.pdf")

## loading the document

In [ ]:
def document_loader(file_path : Path):
    file_format = file_path.suffix.lower()
    if file_format == ".pdf":
        docs = PyMuPDFLoader(file_path=str(file_path))
    elif file_format == ".txt":
        docs = TextLoader(file_path=str(file_path), encoding="utf-8")
    else:
        raise ValueError(f"Unsupported document format: {file_format}")
    return docs.load()


In [ ]:
d = document_loader(file_path=file_path)

## Parent Chunk

In [ ]:
def parent_splitter(documents : List[Document], chunk_size : int, chunk_overlap : int):
        splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap = chunk_overlap)
        parent_chunks = splitter.split_documents(documents)
        return parent_chunks

    

In [ ]:
parent_chunks = parent_splitter(d, 2000, 200)

## children splitter


In [ ]:
def children_splitter(parent_chunks : List[Document], chunk_size : int, chunk_overlap : int):
    splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap = chunk_overlap)
    child_chunks = splitter.split_documents(parent_chunks)
    return child_chunks

In [ ]:
children_splitter(parent_chunks, 200, 20)

## attaching parent - id metadata

In [ ]:
def create_parent_child_mapping(parent_chunks : List[Document]):

    for idx,chunk in enumerate(parent_chunks):
        chunk.metadata["parent_id"] = f"Parent-{idx+1}"
    return parent_chunks,children_splitter(parent_chunks, 200, 20)

In [ ]:
parent_chunks, child_chunks = create_parent_child_mapping(parent_splitter(d, 1000, 100))

In [ ]:
child_chunks[0]

## Parent-Child RAG needs two storage systems. Indoc store for parents and embeddings for children

In [ ]:
# parent store (dictionary)

parent_store = {}

for idx,pchunks in enumerate(parent_chunks):
    parent_store[f"Parent-{idx+1}"] = pchunks.page_content



In [ ]:
# metadata, child_id, parent_id, source, page
from langchain_core.documents import Document
doc = []
for idx,cchunks in enumerate(child_chunks):
    d = {}
    d["child_id"] = idx + 1
    d['source'] =  Path(cchunks.metadata['source']).name
    d['page'] = cchunks.metadata['page']
    d['parent_id'] = cchunks.metadata['parent_id']

    doc.append(Document(
    page_content=cchunks.page_content,
    metadata=d)
)
print(doc)


In [ ]:
from pinecone import Pinecone
from dotenv import load_dotenv
import os
load_dotenv()


pc = Pinecone(api_key="")

index_name = "integrated-dense-py"

if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map":{"text": "chunk_text"}
        }
    )

In [ ]:
index = pc.Index(index_name)

In [ ]:
doc[0]

In [ ]:
records = []

for d in doc:
    records.append({
        "_id": f"child-{d.metadata['child_id']}",
        "chunk_text": d.page_content,
            "parent_id": d.metadata["parent_id"],
            "source": d.metadata["source"],
            "page": d.metadata["page"]
    })

In [ ]:
records[0]

In [ ]:
for i in range(0, len(records), 96):
        batch = records[i:i+96]
        index.upsert_records(namespace="default", records=batch)


# query



In [ ]:
results = index.search(
    namespace="default",
    query={
        "inputs": {"text": "LEAN is"},
        "top_k": 5
    }
)

parent_ids = set()
context_chunks = []
sources = []

for result in results.result.hits:

    pid = result.fields["parent_id"]

    if pid not in parent_ids:
        parent_ids.add(pid)
        context_chunks.append(parent_store[pid])
    
    sources.append({
            "source": result.fields["source"],
            "page": result.fields["page"]
        })

context = "\n\n".join(context_chunks)

print(context[:500])
print(sources)

In [ ]:
from src.chunking.parent_child import ingest

In [ ]:
from pathlib import Path

In [ ]:
file_path = Path(r"C:\Users\amanm\Desktop\learning\rag-project-2\data\2506.08276.pdf")

In [ ]:
records, parents = ingest(file_path=file_path)

In [ ]:
records[0]